# 🎬 DARK FILES — Complete YouTube Production System
### True Crime · Classified Secrets · Dark Chapters of History
---
**This notebook has TWO systems — both completely free:**

**🔍 SYSTEM 1 — Trend Intelligence (Run first)**
- Scans Google Trends, YouTube & Reddit simultaneously
- Finds what people are actively searching right now
- Scores each topic by opportunity (high demand, low competition)
- Detects the unique angle nobody has covered
- Generates complete metadata: titles, description, tags, thumbnail text

**🎬 SYSTEM 2 — Video Generator (After you have your script)**
- Generates realistic motion video per scene (LTX-Video)
- Professional deep voiceover (Microsoft Edge neural TTS)
- Dark ambient background music (Meta MusicGen)
- Cinematic fade transitions + synced captions
- Dark Files color grade (cold blue, crushed blacks, film grain)
- Exports YouTube-ready MP4

---
### ⚡ Before you start
1. Click **Runtime → Change runtime type → T4 GPU → Save**
2. **Run System 1 first** to find your episode topic
3. **Research the unique angle** using Claude
4. **Paste your script into Cell 8** and run System 2

> ⏱️ Trend scan: ~3 min | Video generation: ~30–50 min on free T4 GPU


In [ ]:
# ── SYSTEM 1 · STEP 1: Install Trend Intelligence packages ───────
import subprocess
print("📦  Installing Trend Intelligence packages...")
subprocess.run(['pip', 'install', '-q',
    'pytrends',
    'youtube-search-python',
    'requests',
    'beautifulsoup4',
], check=True)
print("✅  Done!")


In [ ]:
# DARK FILES TREND INTELLIGENCE SYSTEM v2.0
# Scans: Google Trends + YouTube + Reddit
# Output: Ranked topic opportunities + Full metadata package
import json, time, re, random
from datetime import datetime
from collections import Counter
import requests

DF_KEYWORDS = [
    'true crime', 'unsolved mystery', 'classified documents',
    'government cover up', 'missing persons case', 'cold case',
    'serial killer documentary', 'declassified files',
    'unexplained disappearance', 'dark history',
]

DF_SUBREDDITS = [
    'UnresolvedMysteries', 'TrueCrime', 'conspiracy',
    'Missing411', 'ColdCases', 'ClassifiedDocuments',
    'Paranormal', 'MurderMystery',
]

# ── 1. REDDIT SCANNER (no API key needed) ────────────────────────
def scan_reddit():
    print("  Scanning Reddit trending posts...")
    topics = []
    headers = {'User-Agent': 'DarkFilesResearch/2.0'}
    for sub in DF_SUBREDDITS:
        try:
            r = requests.get(
                f'https://www.reddit.com/r/{sub}/hot.json?limit=20',
                headers=headers, timeout=10
            )
            if r.status_code == 200:
                for p in r.json()['data']['children']:
                    d = p['data']
                    if d['score'] > 50 and not d.get('stickied'):
                        topics.append({
                            'title'    : d['title'],
                            'score'    : d['score'],
                            'comments' : d['num_comments'],
                            'subreddit': sub,
                            'url'      : f"https://reddit.com{d['permalink']}",
                            'source'   : 'Reddit',
                        })
            time.sleep(0.4)
        except Exception as e:
            print(f"    Warning r/{sub}: {str(e)[:40]}")
    topics.sort(key=lambda x: x['score'] + x['comments'] * 3, reverse=True)
    print(f"    {len(topics)} trending posts found")
    return topics

# ── 2. GOOGLE TRENDS SCANNER ────────────────────────────────────
def scan_google_trends():
    print("  Scanning Google Trends...")
    trend_scores, rising_topics = {}, []
    try:
        from pytrends.request import TrendReq
        pytrends = TrendReq(hl='en-US', tz=0, timeout=(10, 30))
        batches = [DF_KEYWORDS[i:i+5] for i in range(0, len(DF_KEYWORDS), 5)]
        for batch in batches:
            try:
                pytrends.build_payload(batch, timeframe='now 7-d', geo='')
                df = pytrends.interest_over_time()
                if not df.empty:
                    for kw in batch:
                        if kw in df.columns:
                            vals = df[kw].values
                            trend_scores[kw] = {
                                'avg'     : int(vals.mean()),
                                'peak'    : int(vals.max()),
                                'velocity': int(vals[-1]) - int(vals[0]),
                            }
                time.sleep(1.5)
            except Exception as e:
                print(f"    Trends batch warning: {str(e)[:40]}")
        try:
            pytrends.build_payload(['true crime', 'unsolved mystery'], timeframe='now 7-d')
            related = pytrends.related_queries()
            for kw in related:
                if related[kw].get('rising') is not None:
                    rising_topics += related[kw]['rising']['query'].head(5).tolist()
        except Exception:
            pass
        print(f"    {len(trend_scores)} keyword trends + {len(rising_topics)} rising queries found")
    except Exception as e:
        print(f"    Google Trends limited: {str(e)[:50]}")
    return trend_scores, list(set(rising_topics))

# ── 3. YOUTUBE COMPETITION ANALYZER ─────────────────────────────
def analyze_youtube(topic):
    try:
        from youtubesearchpython import VideosSearch
        results = VideosSearch(f"{topic} documentary", limit=10).result()
        videos  = results.get('result', [])
        titles  = [v.get('title','') for v in videos]
        views   = []
        for v in videos:
            vc   = v.get('viewCount', {})
            text = (vc.get('text','0') if isinstance(vc, dict) else str(vc))
            text = re.sub(r'[^0-9]', '', text)
            try: views.append(int(text))
            except: pass
        avg_views = int(sum(views)/len(views)) if views else 0
        comp      = 'HIGH' if len(videos) >= 8 else ('MEDIUM' if len(videos) >= 4 else 'LOW')
        return {'count': len(videos), 'avg_views': avg_views, 'titles': titles, 'competition': comp}
    except Exception:
        return {'count': 0, 'avg_views': 0, 'titles': [], 'competition': 'UNKNOWN'}

# ── 4. UNIQUE ANGLE ENGINE ───────────────────────────────────────
ANGLE_TEMPLATES = [
    "the classified evidence that was immediately sealed",
    "the witness statement that contradicts the official timeline",
    "the government document released under FOIA that changes everything",
    "the detail every mainstream outlet buried",
    "the investigator who was removed before they could testify",
    "the second victim nobody talks about",
    "the agency that was present but never mentioned in the report",
    "the 48-hour window the official story cannot account for",
    "the forensic anomaly the coroner flagged but was overruled",
    "the survivor who gave one interview and was never heard from again",
    "the files that were destroyed three days before the trial",
    "the location connection every documentary ignored",
]

HOOK_TEMPLATES = [
    "Every documentary told you the story. Nobody told you {a}.",
    "Mainstream media covered the case. They forgot to mention {a}.",
    "The official report runs 847 pages. {a} appears nowhere in it.",
    "Three networks. Four documentaries. Zero mention of {a}.",
    "You have seen the headlines. Here is {a} -- the part they cut.",
]

def generate_angle(topic, yt_titles):
    title_blob = ' '.join(yt_titles).lower()
    covered = []
    if any(w in title_blob for w in ['solved','caught','arrested','found']):
        covered.append("the resolution")
    if any(w in title_blob for w in ['story','explained','happened','truth']):
        covered.append("the surface narrative")
    if any(w in title_blob for w in ['documentary','investigation','case']):
        covered.append("the official investigation")
    angle = random.choice(ANGLE_TEMPLATES)
    hook  = random.choice(HOOK_TEMPLATES).replace('{a}', angle)
    return {
        'what_mainstream_covered': covered or ["the official story"],
        'dark_files_angle'       : angle,
        'opening_hook'           : hook,
    }

# ── 5. SCRIPT OUTLINE GENERATOR ──────────────────────────────────
def generate_outline(topic, angle_data):
    angle = angle_data['dark_files_angle']
    hook  = angle_data['opening_hook']
    lines = [
        f"DARK FILES SCRIPT OUTLINE -- {topic.upper()}",
        "",
        "[00:00 - HOOK]",
        hook,
        "Open with the most disturbing suppressed detail. No context yet.",
        "",
        "[01:30 - SETUP]",
        "Walk through the official narrative calmly. Specific dates, names, locations.",
        "",
        f"[05:00 - THE BURIED DETAIL]",
        f"Introduce: {angle}",
        "Present the evidence. Be specific. Cite sources.",
        "",
        "[10:00 - THE PATTERN]",
        "Show the pattern. Other cases. Other files. Connect the dots.",
        "",
        "[16:00 - WHAT IT MEANS]",
        f"What does {angle} tell us about what really happened?",
        "Present logical conclusion from evidence only. No speculation.",
        "",
        "[20:00 - SIGN OFF]",
        '"The file is still open. Subscribe to Dark Files."',
        "",
        "CLAUDE RESEARCH PROMPT:",
        f'"Research {angle} related to {topic}.',
        "Find specific facts, dates, names and source documents",
        'that mainstream media did not cover. Information gain only."',
    ]
    return '\n'.join(lines)

# ── 6. METADATA GENERATOR ───────────────────────────────────────
def generate_metadata(topic, angle_data, trend_score=50):
    t     = topic.title()
    angle = angle_data['dark_files_angle']

    titles = [
        f"The {t}: {angle.title()} -- Dark Files",
        f"What Every Documentary Got Wrong About {t}",
        f"{t}: The File They Never Wanted Public",
        f"Declassified: {t} & {angle.title()}",
        f"The REAL {t} -- {angle.title()} (Classified)",
    ]

    description = (
        f"DARK FILES -- {t}\n\n"
        f"{angle_data['opening_hook']}\n\n"
        f"In this episode, Dark Files uncovers {angle} surrounding {t}. "
        "Every mainstream documentary covered the surface story. "
        "We go deeper -- into the evidence buried, the witnesses silenced "
        "and the files sealed.\n\n"
        "This is information gain. Details no other channel has covered.\n\n"
        "Sources and documents referenced in this video are cited below.\n\n"
        "CHAPTERS\n"
        "00:00 -- The Detail Nobody Covered\n"
        "01:30 -- What You Were Told\n"
        f"05:00 -- {angle.title()}\n"
        "10:00 -- The Pattern\n"
        "16:00 -- What It Means\n"
        "20:00 -- The File Stays Open\n\n"
        "Subscribe to DARK FILES -- classified content every week.\n"
        f"#DarkFiles #TrueCrime #ClassifiedSecrets #{t.replace(' ','')} "
        "#UnsolvedMystery #Documentary #Conspiracy #Declassified #ColdCase"
    )

    tags = [
        topic.lower(), f"{topic.lower()} documentary", f"{topic.lower()} true crime",
        f"{topic.lower()} unsolved", f"{topic.lower()} explained", f"{topic.lower()} 2026",
        "true crime", "unsolved mystery", "classified documents", "dark files",
        "government cover up", "conspiracy", "declassified", "cold case",
        "missing persons", "dark history", "secret history", "suppressed evidence",
        "what really happened", "documentary 2026", "true crime documentary",
        "scary true stories", "classified secrets", "fbi files", "cia documents",
    ]

    return {
        'best_title'   : titles[0],
        'title_options': titles,
        'description'  : description,
        'tags'         : tags,
        'thumbnail'    : {
            'main' : t.upper(),
            'sub'  : 'THEY HID THIS',
            'style': 'Black bg, blood-red accent, white bold text, FBI redaction stamp',
        },
        'upload_day'   : 'Thursday or Friday',
        'upload_time'  : '6PM - 9PM viewer local time',
        'trend_score'  : trend_score,
    }

# ── 7. OPPORTUNITY SCORER ────────────────────────────────────────
def score_opp(reddit_engagement, trend_avg, trend_velocity, competition):
    reddit_pts   = min(reddit_engagement / 2000 * 30, 30)
    trend_pts    = min(trend_avg / 100 * 25, 25)
    velocity_pts = min(max(trend_velocity, 0) / 50 * 15, 15)
    comp_pts     = {'LOW': 30, 'MEDIUM': 18, 'HIGH': 8, 'UNKNOWN': 12}.get(competition, 12)
    return int(reddit_pts + trend_pts + velocity_pts + comp_pts)

# ── 8. MASTER SCAN ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("DARK FILES TREND INTELLIGENCE SYSTEM")
print("Scanning: Google Trends | YouTube | Reddit")
print("=" * 65 + "\n")

reddit_topics         = scan_reddit()
trend_data, rising    = scan_google_trends()

print("  Analyzing YouTube competition + generating angles...")
opportunities = []

for post in reddit_topics[:10]:
    topic = re.sub(r'[^a-zA-Z0-9 ]', '', post['title'])[:70].strip()
    if len(topic) < 8:
        continue
    yt       = analyze_youtube(topic)
    angle    = generate_angle(topic, yt['titles'])
    t_info   = trend_data.get(topic.lower().split()[0], {})
    t_avg    = t_info.get('avg', 20)
    t_vel    = t_info.get('velocity', 0)
    opp      = score_opp(post['score'], t_avg, t_vel, yt['competition'])
    opportunities.append({
        'topic'       : topic,
        'source'      : f"Reddit r/{post['subreddit']}",
        'engagement'  : post['score'],
        'opportunity' : opp,
        'competition' : yt['competition'],
        'trend_vel'   : t_vel,
        'angle'       : angle,
        'metadata'    : generate_metadata(topic, angle, t_avg),
        'outline'     : generate_outline(topic, angle),
        'yt_count'    : yt['count'],
        'yt_avg_views': yt['avg_views'],
    })
    time.sleep(0.3)

for rt in rising[:4]:
    if len(rt) < 6:
        continue
    yt    = analyze_youtube(rt)
    angle = generate_angle(rt, yt['titles'])
    opp   = score_opp(600, 75, 40, yt['competition'])
    opportunities.append({
        'topic': rt, 'source': 'Google Trends (Rising)', 'engagement': 600,
        'opportunity': opp, 'competition': yt['competition'], 'trend_vel': 40,
        'angle': angle, 'metadata': generate_metadata(rt, angle, 75),
        'outline': generate_outline(rt, angle),
        'yt_count': yt['count'], 'yt_avg_views': yt['avg_views'],
    })
    time.sleep(0.3)

opportunities.sort(key=lambda x: x['opportunity'], reverse=True)

# ── DISPLAY RESULTS ──────────────────────────────────────────────
print("\n" + "=" * 65)
print("TOP DARK FILES EPISODE OPPORTUNITIES THIS WEEK")
print("=" * 65)

for i, o in enumerate(opportunities[:5]):
    s   = o['opportunity']
    vel = o['trend_vel']
    tag = "HOT"    if s >= 70 else ("STRONG" if s >= 50 else "SOLID")
    trn = "Rising" if vel > 5 else ("Stable" if vel >= -5 else "Fading")

    print(f"\n{'─'*65}")
    print(f"[{tag}] #{i+1} -- OPPORTUNITY SCORE: {s}/100  |  Trend: {trn}")
    print(f"{'─'*65}")
    print(f"TOPIC       : {o['topic']}")
    print(f"SOURCE      : {o['source']}")
    print(f"ENGAGEMENT  : {o['engagement']:,} Reddit score")
    print(f"COMPETITION : {o['competition']} on YouTube ({o['yt_count']} existing videos)")
    print(f"AVG VIEWS   : {o['yt_avg_views']:,} on competing videos")
    print(f"\nUNIQUE ANGLE (what nobody covered):")
    print(f"  -> {o['angle']['dark_files_angle']}")
    print(f"\nOPENING HOOK:")
    print(f'  "{o["angle"]["opening_hook"]}"')
    print(f"\nTITLE OPTIONS:")
    for j, t in enumerate(o['metadata']['title_options'][:3], 1):
        print(f"  {j}. {t}")
    print(f"\nTOP TAGS    : {', '.join(o['metadata']['tags'][:7])}")
    print(f"THUMBNAIL   : [{o['metadata']['thumbnail']['main']}] [{o['metadata']['thumbnail']['sub']}]")
    print(f"UPLOAD      : {o['metadata']['upload_day']} at {o['metadata']['upload_time']}")

print("\n" + "=" * 65)
print("\nSCRIPT OUTLINE -- Top Opportunity:")
print(opportunities[0]['outline'] if opportunities else "No topics found.")
print("""
===============================================================
NEXT STEPS:
  1. Pick the topic with the highest score above
  2. Use Claude to research the UNIQUE ANGLE
     (paste the CLAUDE RESEARCH PROMPT from the outline)
  3. Write your full Dark Files script (500+ words)
  4. Scroll down to Cell 8, paste your script
  5. Runtime -> Run all -> Download your finished video
===============================================================
""")


In [ ]:
# ── STEP 0: Verify GPU ──────────────────────────────────────────
import subprocess, os, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print("❌  No GPU found!")
    print("👉  Go to Runtime → Change runtime type → GPU → T4 → Save")
    print("    Then restart and run again.")
    sys.exit(1)

# Print GPU name and memory
for line in result.stdout.split('\n'):
    if 'Tesla' in line or 'T4' in line or 'A100' in line or 'L4' in line or 'V100' in line:
        print(f"✅  GPU: {line.strip()}")
        break
else:
    print("✅  GPU detected")
    print(result.stdout[:300])

# Create working directories
for d in ['/content/dark_files/clips', '/content/dark_files/audio',
          '/content/dark_files/final', '/content/dark_files/cache']:
    os.makedirs(d, exist_ok=True)

print("\n📁  Working directories ready")
print("🚀  Ready to generate Dark Files videos!")


In [ ]:
# ── STEP 1: Install dependencies ────────────────────────────────
print("📦  Installing packages (3–5 min first time)...")

import subprocess
pkgs = [
    "edge-tts",
    "diffusers>=0.31.0",
    "transformers>=4.40.0",
    "accelerate",
    "imageio[ffmpeg]",
    "moviepy",
    "scipy",
    "sentencepiece",
    "protobuf",
]
subprocess.run(
    ["pip", "install", "-q", "--upgrade"] + pkgs,
    check=True
)

# Verify FFmpeg
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(f"✅  {r.stdout.splitlines()[0]}" if r.returncode == 0 else "❌  FFmpeg missing")
print("✅  All packages installed!")


In [ ]:
# ── STEP 2: Imports ─────────────────────────────────────────────
import os, re, json, asyncio, subprocess, math, time, warnings
import numpy as np
from pathlib import Path

import torch
warnings.filterwarnings('ignore')

from IPython.display import Video, display, HTML
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"🖥️   GPU : {props.name}")
    print(f"💾  VRAM : {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️   Running on CPU — video generation will be very slow")

print(f"\n✅  Ready  |  Device: {DEVICE.upper()}")


In [ ]:
# ── STEP 3: Episode Settings & Upload Script ─────────────────────
# 1. Edit EPISODE_TITLE and voice below.
# 2. Run this cell.
# 3. Click Choose Files and select your script as a plain .txt file.
#    Your script can contain any characters (dashes, quotes, etc.)
# ─────────────────────────────────────────────────────────────────

EPISODE_TITLE  = "The Classified Files"   # <- change this

# Voice options (uncomment one)
VOICE          = "en-US-GuyNeural"        # Deep authoritative male  <- DEFAULT
# VOICE        = "en-US-EricNeural"       # Serious male narrator
# VOICE        = "en-US-ChristopherNeural"# Calm, grave male voice
# VOICE        = "en-GB-RyanNeural"       # British male (BBC documentary feel)

SPEAKING_RATE  = "-10%"
SPEAKING_PITCH = "-3Hz"

# ── Upload script ────────────────────────────────────────────────
from google.colab import files as _gcf
print(f"Episode: {EPISODE_TITLE}")
print(f"Voice  : {VOICE}")
print("\nClick Choose Files below and select your script as a .txt file...")
_up = _gcf.upload()
if not _up:
    raise RuntimeError("No file uploaded. Run this cell again.")
YOUR_SCRIPT = list(_up.values())[0].decode("utf-8", errors="replace").strip()
words = len(YOUR_SCRIPT.split())
print(f"\nScript loaded: {words} words  (~{words / (110/60):.0f}s  {words / (110/60) / 60:.1f} min)")


In [ ]:
# ── STEP 4: Scene parser + Dark Files prompt engine ─────────────

# ── Dark Files visual prompt database ───────────────────────────
DARK_AESTHETIC = (
    "award winning cinematic photography, professional Hollywood cinematography, "
    "dramatic three-point lighting, rich deep colors, sharp crisp focus, "
    "photorealistic ultra detailed 8k, beautiful atmospheric depth, "
    "teal and orange color grade, visible shadow detail, "
    "National Geographic documentary style, stunning visual composition"
)

DARK_NEGATIVE = (
    "cartoon, animated, illustration, painting, drawing, sketch, "
    "blurry, out of focus, low quality, distorted, watermark, text overlay, logo, nsfw, "
    "overexposed, washed out, flat lighting, stock photo, generic, boring, "
    "duplicate, deformed, ugly, bad anatomy, grainy, noisy, pixelated, "
    "pitch black, too dark, underexposed, muddy, unclear"
)

VISUAL_MAP = [
    (r'pilot|cockpit|cessna|aviator|flying|took off|takeoff|altitude|feet',
     'stunning cinematic cockpit interior at dusk, young pilot in uniform with focused expression, beautifully lit instrument panel with amber and blue glowing dials, vast sky visible through windscreen, cinematic depth of field, dramatic side lighting'),
    (r'radio|transmission|signal|contact|broadcast|frequency|transmit|microphone',
     'cinematic close-up of vintage 1970s radio equipment, glowing frequency dials in amber and green, audio waveform display, dramatic warm side lighting, rich textures, professional photography'),
    (r'air.?traffic|controller|tower|radar|robey|melbourne',
     'cinematic air traffic control room at night, multiple glowing radar screens in teal and green, professional controller at workstation, dramatic blue-amber lighting, rich atmospheric depth'),
    (r'metallic|scraping|sound|noise|interference|silence|static',
     'cinematic close-up of reel-to-reel tape recorder with audio waveform display, warm amber instrument lighting, dramatic side shadows, rich detailed textures, 1970s professional equipment'),
    (r'search|rescue|vessel|coastguard|aircraft.*search|search.*aircraft',
     'dramatic cinematic wide shot of coastguard search vessel on vast open ocean at sunset, powerful searchlights, golden and teal water reflections, breathtaking atmospheric photography'),
    (r'photograph|photo|camera|image|picture|manifold',
     'cinematic close-up of old classified photographs spread on a wooden desk, dramatic warm side lighting, magnifying glass, visible film grain texture on photos, rich vintage atmosphere'),
    (r'ocean|sea|bass strait|water|coast|strait|overwater',
     'breathtaking cinematic wide shot of vast ocean at dusk, dramatic golden-teal sky reflected on water surface, storm clouds building on horizon, stunning atmospheric photography'),
    (r'forest|woods|trees|woodland',
     'stunning cinematic forest scene, shafts of golden light through ancient tall trees, beautiful atmospheric mist between trunks, rich green and amber tones, breathtaking nature photography'),
    (r'facility|complex|plant|factory|warehouse',
     'cinematic wide shot of industrial facility at dramatic dusk, warm orange sunset behind steel structures, security lighting, rich orange-teal color contrast, architectural photography'),
    (r'laborator|lab|research|scientist',
     'cinematic government laboratory interior, clean white sterile environment, dramatic blue fluorescent lighting, scientist silhouette at equipment, rich color contrast, professional photography'),
    (r'city|town|street|urban|neighborhood|melbourne',
     'stunning cinematic city street at blue hour, rain-wet pavement reflecting neon signs in amber and teal, beautiful bokeh lights in background, dramatic atmospheric photography'),
    (r'desert|nevada|arizona|wasteland|remote|plains',
     'breathtaking cinematic desert landscape at golden hour, dramatic clouds casting long shadows, desolate road to horizon, rich warm orange and purple tones, stunning wide shot'),
    (r'prison|jail|detention|cell|incarcerat',
     'cinematic prison corridor with dramatic lighting, iron bars casting long geometric shadows, warm overhead light contrasting with cool blue shadows, rich architectural photography'),
    (r'courtroom|trial|judge|lawyer|testimony|verdict',
     'dramatic cinematic courtroom interior, oak paneling, warm overhead spotlights on witness stand, beautiful architectural details, rich amber and shadow contrast, atmospheric photography'),
    (r'cemetery|grave|tombstone|burial',
     'cinematic cemetery at dramatic blue hour, beautiful old stone monuments, mist rolling between headstones, rich teal and amber light, atmospheric documentary photography'),
    (r'hospital|medical|morgue|autopsy|clinic',
     'cinematic hospital corridor, dramatic perspective with vanishing point, fluorescent blue-white lighting, reflective floors, atmospheric depth, professional architectural photography'),
    (r'police|detective|investig|crime scene|sheriff',
     'cinematic crime scene photograph, dramatic yellow tape against dark background, red-blue police lights reflecting on wet pavement, atmospheric depth, professional documentary photography'),
    (r'document|file|report|classif|declassif|evidence|record|freedom.?of.?information|foia|sealed',
     'stunning cinematic close-up of declassified documents with redacted black lines, dramatic warm side lighting, rich paper texture, wooden desk surface, atmospheric documentary style'),
    (r'disappear|vanish|missing|abduct|gone|never.*found|never.*seen',
     'cinematic empty room with dramatic single window light, overturned chair casting long shadow, beautiful atmospheric dust particles in light beam, rich contrast, haunting composition'),
    (r'government|military|pentagon|CIA|FBI|NSA|agency|federal|intelligence|department',
     'dramatic cinematic wide shot of imposing government building at dawn, beautiful warm light on stone facade, symmetrical architectural composition, rich teal sky, stunning photography'),
    (r'witness|survivor|victim|family|father|mother|guido',
     'cinematic portrait silhouette of person at large window, beautiful blue-hour light from outside, rich rim lighting, thoughtful composition, atmospheric documentary photography'),
    (r'secret|hidden|cover|buried|suppress|conceal',
     'cinematic close-up of heavy vault or safe door, dramatic warm side lighting, rich metal textures, combination lock detail, beautiful chiaroscuro shadows, professional photography'),
    (r'helicopter|military.*aircraft|search.*plane',
     'dramatic cinematic shot of helicopter silhouetted against stunning sunset sky, rotor blur, rich orange and purple clouds, breathtaking wide angle, professional aviation photography'),
    (r'phone|call|wiretap|surveillance|listen|intercept',
     'cinematic close-up of vintage rotary telephone on dark wooden desk, warm amber desk lamp, beautiful shallow depth of field, rich textures, 1970s atmospheric photography'),
    (r'newspaper|media|press|headline|journalist|editor',
     'cinematic newspaper archive, stacked yellowed papers with visible headlines, warm single lamp overhead, beautiful vintage atmosphere, rich amber tones, documentary photography'),
    (r'night|midnight|dark|evening|dusk|1978|october',
     'breathtaking cinematic night sky full of stars over calm water, milky way reflection, rich blue and silver tones, stunning astrophotography style, dramatic atmospheric wide shot'),
    (r'road|highway|bridge|airport|runway|moorabbin',
     'stunning cinematic airport runway at blue hour, perspective lines of runway lights stretching to horizon, dramatic teal sky, aircraft silhouette, beautiful aviation photography'),
    (r'body|remains|skeleton|bones|buried',
     'cinematic forensic scene at dusk, professional lighting equipment casting warm glow, evidence markers, dramatic atmospheric photography, rich color contrast'),
    (r'national.?security|classified|sealed|denied|withheld',
     'dramatic cinematic close-up of TOP SECRET stamp on document, rich red ink on aged paper, warm dramatic side lighting, beautiful depth of field, atmospheric documentary photography'),
    (r'search.*four|four.*day|called.*off|terminated|abandoned',
     'cinematic wide shot of empty ocean horizon at dawn, search vessel turning back, dramatic golden light, vast scale of ocean, heartbreaking beautiful composition'),
    (r'transcript|words|said|spoke|voice|last.*word',
     'cinematic close-up of typed transcript paper, individual words in sharp focus, warm side lighting, dramatic shadow from page edge, rich texture detail, documentary photography'),
]

def parse_scenes(script, max_words=55):
    paras = [p.strip() for p in script.strip().split('\n') if p.strip()]
    scenes = []
    for para in paras:
        if len(para.split()) <= max_words:
            scenes.append(para)
        else:
            sents = re.split(r'(?<=[.!?])\s+', para)
            bucket, bw = [], 0
            for s in sents:
                sw = len(s.split())
                if bw + sw > max_words and bucket:
                    scenes.append(' '.join(bucket))
                    bucket, bw = [s], sw
                else:
                    bucket.append(s); bw += sw
            if bucket:
                scenes.append(' '.join(bucket))
    return [s for s in scenes if s.strip()]

def make_prompt(scene_text):
    low = scene_text.lower()
    matched = []
    for pattern, visual in VISUAL_MAP:
        if re.search(pattern, low, re.IGNORECASE):
            matched.append(visual)
    if matched:
        base = ', '.join(matched[:2])
    else:
        base = 'dark mysterious empty environment, dramatic single light source, heavy shadows, fog'
    return f"{base}, {DARK_AESTHETIC}", DARK_NEGATIVE

def estimate_secs(text, wpm=110):
    return (len(text.split()) / wpm) * 60

# ── Preview ──────────────────────────────────────────────────────
scenes = parse_scenes(YOUR_SCRIPT)
print(f"📊  Parsed into {len(scenes)} scenes\n{'='*65}")
for i, s in enumerate(scenes):
    dur = estimate_secs(s)
    prompt, _ = make_prompt(s)
    print(f"\n🎬  Scene {i+1}  ({dur:.1f}s | {len(s.split())} words)")
    print(f"    Text   : {s[:75]}...")
    print(f"    Prompt : {prompt[:80]}...")
total = sum(estimate_secs(s) for s in scenes)
print(f"\n{'='*65}")
print(f"⏱️   Total estimated duration: {total:.0f}s  ({total/60:.1f} min)")
print(f"⏳  Estimated generation time on T4: {len(scenes)*3:.0f}–{len(scenes)*5:.0f} min")


In [ ]:
# ── STEP 5: Generate voiceover + caption timing ──────────────────
import nest_asyncio
nest_asyncio.apply()
import edge_tts, asyncio, json, subprocess

async def gen_voice_simple(text, voice, rate, pitch, audio_out):
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
    await communicate.save(audio_out)

def audio_dur(path):
    r = subprocess.run(
        ['ffprobe','-v','quiet','-print_format','json','-show_format', path],
        capture_output=True, text=True
    )
    return float(json.loads(r.stdout)['format']['duration'])

print(f"🎙️   Generating voiceover  ({VOICE})...\n")
audio_paths, vtt_paths, durations = [], [], []

for i, scene in enumerate(scenes):
    ap = f'/content/dark_files/audio/scene_{i:03d}.mp3'
    vp = f'/content/dark_files/audio/scene_{i:03d}.vtt'
    asyncio.run(gen_voice_simple(scene, VOICE, SPEAKING_RATE, SPEAKING_PITCH, ap))
    d = audio_dur(ap)
    with open(vp, 'w') as f:
        f.write(f"WEBVTT\n\n00:00.000 --> {int(d//60):02d}:{d%60:06.3f}\n{scene}\n\n")
    audio_paths.append(ap); vtt_paths.append(vp); durations.append(d)
    print(f"  ✅  Scene {i+1}/{len(scenes)}: {d:.2f}s  |  {scene[:55]}...")

total_duration = sum(durations)
print(f"\n✅  Voiceover complete!")
print(f"⏱️   Total duration: {total_duration:.1f}s  ({total_duration/60:.1f} min)")


In [ ]:
# ── STEP 6: Load Stable Diffusion XL image model ────────────────
#
#  SDXL: ~7 GB download — high quality cinematic images on T4 GPU.
#  Generates one detailed scene image per paragraph; FFmpeg adds
#  Ken Burns pan/zoom motion to create smooth video clips.
# ─────────────────────────────────────────────────────────────────
import torch
from diffusers import StableDiffusionXLPipeline

torch.cuda.empty_cache()
print("Loading Stable Diffusion XL (~7 GB — first run only, please wait)...\n")

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
    cache_dir='/content/dark_files/cache',
)
pipe.enable_model_cpu_offload()

print(f"\n✅  Stable Diffusion XL loaded!")
print(f"    VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB / 16 GB")


In [ ]:
# ── STEP 7: Generate cinematic images + Ken Burns clips (audio-synced) ───────
#
#  SDXL generates one dark cinematic image per scene.
#  Each clip embeds its own voiceover audio via -shortest so the video
#  is mathematically exactly as long as the audio — no drift, no lag.
# ─────────────────────────────────────────────────────────────────────────────
import time

KEN_BURNS = [
    "zoompan=z='min(zoom+0.0015,1.5)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)'",
    "zoompan=z='min(zoom+0.0012,1.4)':x='min(iw-iw/zoom,iw/2-(iw/zoom/2)+in*0.4)':y='ih/2-(ih/zoom/2)'",
    "zoompan=z='min(zoom+0.0012,1.4)':x='iw/2-(iw/zoom/2)':y='max(0,ih/2-(ih/zoom/2)-in*0.3)'",
    "zoompan=z='max(1.0,1.5-in*0.0018)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)'",
    "zoompan=z='1.35':x='min(iw-iw/zoom,in*0.6)':y='ih/2-(ih/zoom/2)'",
]

def gen_image(prompt, neg, img_path, seed=0):
    gen = torch.Generator(device='cuda').manual_seed(seed)
    img = pipe(
        prompt=prompt,
        negative_prompt=neg,
        width=1024, height=576,
        num_inference_steps=25,
        guidance_scale=7.5,
        generator=gen,
    ).images[0]
    img.save(img_path)

def image_to_clip(img_path, audio_path, out_path, duration, effect_idx=0):
    fps    = 25
    frames = max(int(duration * fps) + 1, 2)
    effect = KEN_BURNS[effect_idx % len(KEN_BURNS)]
    # Embed voiceover audio in the clip. -shortest cuts video to exact audio length.
    subprocess.run([
        'ffmpeg', '-y',
        '-loop', '1', '-i', img_path,
        '-i', audio_path,
        '-filter_complex', f'[0:v]{effect}:d={frames}:s=1024x576,fps={fps}[v]',
        '-map', '[v]', '-map', '1:a',
        '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
        '-c:a', 'aac', '-b:a', '192k',
        '-pix_fmt', 'yuv420p', '-shortest',
        out_path
    ], capture_output=True, check=True)

print(f"Generating {len(scenes)} cinematic clips ...\n{'='*65}")
clip_paths = []
t0 = time.time()

for i, (scene, dur, ap) in enumerate(zip(scenes, durations, audio_paths)):
    img_path = f'/content/dark_files/clips/scene_{i:03d}.png'
    out_path = f'/content/dark_files/clips/clip_{i:03d}.mp4'
    prompt, neg = make_prompt(scene)

    print(f"\nScene {i+1}/{len(scenes)} ({dur:.1f}s)")
    print(f"    {prompt[:85]}...")

    t1 = time.time()
    gen_image(prompt, neg, img_path, seed=i * 17 + 42)
    image_to_clip(img_path, ap, out_path, dur, effect_idx=i)
    clip_paths.append(out_path)

    elapsed = time.time() - t1
    done    = i + 1
    eta     = (time.time() - t0) / done * (len(scenes) - done)
    print(f"    Done: {elapsed:.0f}s  |  ETA: ~{eta/60:.0f} min remaining")

print(f"\nAll clips done! Total: {(time.time()-t0)/60:.1f} min")


In [ ]:
# ── STEP 8: Generate dark ambient background music ───────────────
from transformers import pipeline as hf_pipeline
import wave, numpy as np

# Free GPU memory from image model first
del pipe
torch.cuda.empty_cache()

print("Loading MusicGen-small ...")
music_pipe = hf_pipeline(
    'text-to-audio',
    'facebook/musicgen-small',
    device=0 if DEVICE == 'cuda' else -1,
)

MUSIC_PROMPT = (
    "dark ambient music, mysterious thriller documentary, tense suspenseful atmosphere, "
    "slow deep bass drones, ominous low-frequency hum, haunting cinematic orchestral score, "
    "psychological thriller, minimal slow tempo, no vocals, instrumental only, eerie"
)

print("Generating dark ambient loop ...")
music = music_pipe(MUSIC_PROMPT, forward_params={"do_sample": True, "max_new_tokens": 1500})

# Handle both list and dict output formats across transformers versions
audio_data = music[0] if isinstance(music, list) else music
sr         = audio_data['sampling_rate']
_arr       = audio_data['audio']
if _arr.ndim == 2:
    _arr = _arr.mean(axis=0)   # stereo -> mono
_pcm = (_arr * 32767).clip(-32768, 32767).astype(np.int16)

# Write WAV using built-in wave module — no scipy dependency
raw_wav = '/content/dark_files/audio/music_raw.wav'
with wave.open(raw_wav, 'w') as _wf:
    _wf.setnchannels(1)
    _wf.setsampwidth(2)
    _wf.setframerate(sr)
    _wf.writeframes(_pcm.tobytes())

# Loop to match full video duration + fade out
music_wav = '/content/dark_files/audio/music_final.wav'
fade_start = max(0, total_duration - 4)
subprocess.run([
    'ffmpeg', '-stream_loop', '-1', '-i', raw_wav,
    '-t', str(total_duration + 2),
    '-af', f'afade=t=out:st={fade_start:.1f}:d=4',
    '-c:a', 'pcm_s16le', music_wav, '-y'
], capture_output=True, check=True)

del music_pipe
torch.cuda.empty_cache()
print(f"Music ready ({total_duration:.0f}s, fades out at end)")


In [ ]:
# ── STEP 9: Assemble clips with fade transitions ─────────────────
#  Each clip already carries its own voiceover audio (embedded in Step 7).
#  We apply fades, concatenate, then feed into color grade.

def add_fades(src, dst, dur, fade_dur=0.4):
    fade_out_start = max(0, dur - fade_dur)
    subprocess.run([
        'ffmpeg', '-y', '-i', src,
        '-vf', (f'fade=t=in:st=0:d={fade_dur},'
                f'fade=t=out:st={fade_out_start:.2f}:d={fade_dur}'),
        '-c:v', 'libx264', '-crf', '18', '-preset', 'fast', '-pix_fmt', 'yuv420p',
        '-c:a', 'copy',   # preserve audio stream — already synced
        dst
    ], capture_output=True, check=True)

print("Adding fade transitions to each clip ...")
faded_paths = []
for i, (cp, dur) in enumerate(zip(clip_paths, durations)):
    fp = f'/content/dark_files/clips/clip_{i:03d}_faded.mp4'
    add_fades(cp, fp, dur)
    faded_paths.append(fp)
    print(f"  Clip {i+1}/{len(scenes)} faded")

# Concatenate all faded clips — audio (voiceover) is already embedded per clip
concat_list = '/content/dark_files/concat.txt'
with open(concat_list, 'w') as f:
    for p in faded_paths:
        f.write(f"file '{p}'\n")

video_with_voice = '/content/dark_files/final/video_raw.mp4'
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_list,
    '-c:v', 'libx264', '-crf', '17', '-preset', 'fast', '-pix_fmt', 'yuv420p',
    '-c:a', 'aac', '-b:a', '192k',
    video_with_voice
], capture_output=True, check=True)
print("\nAll clips assembled — voiceover perfectly synced to each scene.")


In [ ]:
# ── STEP 10: Apply Dark Files cinematic color grade ──────────────

print("🎨  Applying Dark Files color grade ...")
print("    → Cold blue channel shift")
print("    → Crushed blacks + compressed highlights")
print("    → Desaturated tones (0.65 saturation)")
print("    → High contrast (1.3)")
print("    → Subtle film grain")

DARK_FILES_GRADE = ",".join([
    # Hollywood teal-orange look: teal in shadows, warm in highlights
    "colorchannelmixer=rr=1.02:rg=0.0:rb=-0.02:gr=-0.01:gg=0.98:gb=0.03:br=-0.02:bg=0.05:bb=1.0",
    # Lift blacks slightly (cinematic) + gentle S-curve
    "curves=r='0/0.02 0.5/0.52 1/0.98':g='0/0.01 0.5/0.50 1/0.99':b='0/0.03 0.5/0.51 1/0.97'",
    # Slight saturation boost, gentle contrast — keep it vivid and beautiful
    "eq=saturation=1.05:contrast=1.08:brightness=0.0:gamma=1.0",
    # Barely-there film grain — just enough for texture
    "noise=alls=1:allf=t",
    # Gentle sharpening for crisp detail
    "unsharp=5:5:0.4:3:3:0.0"
])

video_graded = '/content/dark_files/final/video_graded.mp4'
subprocess.run([
    'ffmpeg', '-i', video_with_voice,
    '-vf', DARK_FILES_GRADE,
    '-c:v', 'libx264', '-crf', '17', '-preset', 'slow',
    '-c:a', 'copy',
    video_graded, '-y'
], check=True, capture_output=True)

print("\n✅  Dark Files color grade applied!")


In [ ]:
# ── STEP 11: Build + burn synced captions ───────────────────────

def vtt_time_to_ms(t):
    t = t.strip().replace(',','.')
    parts = t.split(':')
    if len(parts) == 3:
        h, m, s = parts
        return (int(h)*3600 + int(m)*60 + float(s)) * 1000
    m, s = parts
    return (int(m)*60 + float(s)) * 1000

def ms_to_srt(ms):
    ms = int(ms)
    h  = ms // 3_600_000; ms %= 3_600_000
    m  = ms // 60_000;    ms %= 60_000
    s  = ms // 1_000;     ms %= 1_000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

print("📝  Building synced SRT caption file ...")
entries, idx, offset_ms = [], 1, 0

for vp, dur in zip(vtt_paths, durations):
    try:
        with open(vp, encoding='utf-8') as f:
            content = f.read()
        for block in re.split(r'\n{2,}', content.strip()):
            if '-->' not in block:
                continue
            lines = block.strip().splitlines()
            t_line = next((l for l in lines if '-->' in l), None)
            if not t_line:
                continue
            text = ' '.join(
                l for l in lines
                if '-->' not in l and not l.strip().isdigit() and l.strip()
            )
            if not text:
                continue
            s_str, e_str = t_line.split('-->')
            s_ms = vtt_time_to_ms(s_str) + offset_ms
            e_ms = vtt_time_to_ms(e_str) + offset_ms
            entries.append((idx, ms_to_srt(s_ms), ms_to_srt(e_ms), text))
            idx += 1
    except Exception as ex:
        print(f"  ⚠️   VTT parse warning for {vp}: {ex}")
    offset_ms += dur * 1000

srt_path = '/content/dark_files/audio/captions.srt'
with open(srt_path, 'w', encoding='utf-8') as f:
    for n, s, e, t in entries:
        f.write(f"{n}\n{s} --> {e}\n{t}\n\n")
print(f"  ✅  {len(entries)} caption entries generated")

# Burn captions with Dark Files styling
CAPTION_STYLE = (
    "FontName=Arial,Bold=1,FontSize=17,"
    "PrimaryColour=&H00FFFFFF,"     # white text
    "OutlineColour=&H00000000,"     # black outline
    "BackColour=&H55000000,"        # semi-transparent black box
    "BorderStyle=3,Outline=1.5,Shadow=0,"
    "Alignment=2,MarginV=28"        # bottom centre, 28px margin
)

video_captioned = '/content/dark_files/final/video_captioned.mp4'
subprocess.run([
    'ffmpeg', '-i', video_graded,
    '-vf', f"subtitles={srt_path}:force_style='{CAPTION_STYLE}'",
    '-c:v', 'libx264', '-crf', '17',
    '-c:a', 'copy',
    video_captioned, '-y'
], check=True, capture_output=True)

print("✅  Captions burned in!")


In [ ]:
# ── STEP 12: Final audio mix (voice 100% + music 15%) ───────────

print("🎵  Mixing voiceover + background music ...")

safe_title  = re.sub(r'[^\w\s-]', '', EPISODE_TITLE).strip().replace(' ', '_')
final_video = f'/content/dark_files/final/DARK_FILES_{safe_title}.mp4'

subprocess.run([
    'ffmpeg',
    '-i', video_captioned,
    '-i', music_wav,
    '-filter_complex',
    '[0:a]volume=1.0[v];[1:a]volume=0.14[m];[v][m]amix=inputs=2:duration=first:dropout_transition=2[mix]',
    '-map', '0:v', '-map', '[mix]',
    '-c:v', 'copy',
    '-c:a', 'aac', '-b:a', '192k',
    '-shortest',
    final_video, '-y'
], check=True, capture_output=True)

# ── Final stats ──────────────────────────────────────────────────
r = subprocess.run(
    ['ffprobe','-v','quiet','-print_format','json','-show_format','-show_streams', final_video],
    capture_output=True, text=True
)
info = json.loads(r.stdout)
size_mb  = int(info['format']['size']) / (1024*1024)
vid_dur  = float(info['format']['duration'])
v_stream = next((s for s in info['streams'] if s['codec_type']=='video'), {})
width    = v_stream.get('width', 704)
height   = v_stream.get('height', 480)

print(f"\n{'='*55}")
print(f"🎉  DARK FILES VIDEO COMPLETE!")
print(f"{'='*55}")
print(f"📁  File     : {os.path.basename(final_video)}")
print(f"⏱️   Duration : {vid_dur:.1f}s  ({vid_dur/60:.1f} min)")
print(f"📊  Size     : {size_mb:.1f} MB")
print(f"🎬  Res.     : {width}x{height} @ 25fps")
print(f"🎨  Grade    : Dark Files cinematic (cold blue, high contrast)")
print(f"🎙️   Voice    : {VOICE}")
print(f"📝  Captions : {len(entries)} synced entries")
print(f"🎵  Music    : Dark ambient (AI-generated, royalty-free)")
print(f"{'='*55}")


In [ ]:
# ── STEP 13: Save to Google Drive + download ─────────────────────

import os, shutil
from IPython.display import Video, display
from google.colab import files as colab_files

# ── Mount Google Drive and save a permanent copy ─────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_folder = '/content/drive/MyDrive/DarkFiles'
    os.makedirs(drive_folder, exist_ok=True)
    safe_title = re.sub(r'[^\w\s-]', '', EPISODE_TITLE).strip().replace(' ', '_')
    drive_path  = f'{drive_folder}/DARK_FILES_{safe_title}.mp4'
    shutil.copy2(final_video, drive_path)
    print(f"Saved to Google Drive: DarkFiles/DARK_FILES_{safe_title}.mp4")
    print("Your video is safe — even if Colab disconnects.")
except Exception as e:
    print(f"Drive save skipped: {e}")

# ── Preview ──────────────────────────────────────────────────────
print("\nLoading preview ...\n")
try:
    display(Video(final_video, width=1024, height=576, embed=True))
except Exception:
    pass

# ── Download to your computer ────────────────────────────────────
print("\nDownloading your Dark Files video to your computer ...")
colab_files.download(final_video)

print("""
YOUR VIDEO IS READY FOR YOUTUBE

  MP4 container (H.264 + AAC)
  16:9 aspect ratio  |  25 fps  |  192 kbps audio
  Professional neural voiceover
  Cinematic color grade
  Dark ambient background music (royalty-free)

Your video is also saved permanently in Google Drive
under My Drive > DarkFiles

Pro tips after upload:
  - Add the thumbnail you generated in YouTube Studio
  - Paste the SEO description and tags
  - Add chapter markers for longer episodes
""")
